<a href="https://colab.research.google.com/github/agungdaus/data-science-2026/blob/main/Pertemuan12_Agung_Firdaus_240401010258.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#info

---




```
NAMA  : AGUNG FIRDAUS

NIM   : 240401010258

KELAS : IF403
```



PERTEMUAN 12 :

Clustering (K-Means & Hierarchical)

In [6]:
#step 1
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [4]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

   Gula  Selai   Teh
0  True   True  True


In [ ]:
#step 3
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

In [ ]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
'support', 'confidence', 'lift']].head(10))
# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

**Step 4:**
Aturan dengan Lift tertinggi biasanya adalah **Roti → Selai** (atau sebaliknya),
karena pola ini sengaja disuntikkan di Step 1. Lift > 1 menandakan kemunculan
kedua produk saling berkorelasi positif (tidak kebetulan/independen). Secara
bisnis, ini masuk akal — Roti dan Selai memang produk yang sering dibeli
bersamaan dalam kehidupan nyata (complementary products), sehingga hasil ini
konsisten dengan intuisi bisnis dan bisa dimanfaatkan untuk strategi cross-selling
atau penempatan produk berdekatan di rak toko.

In [5]:
#step 5
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
                 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

In [ ]:
#step 6
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x)]

print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

**Step 6:**
Kedua pendekatan bisa memberi rekomendasi yang **berbeda** karena dasar
logikanya berbeda:
- **Association Rules** merekomendasikan berdasarkan pola pembelian aktual
  (misalnya Roti → Selai, karena history transaksi menunjukkan keduanya
  sering dibeli bersamaan).
- **Content-Based** merekomendasikan berdasarkan kemiripan atribut/kategori
  produk (misalnya Roti → Sereal, karena sama-sama kategori Bakery),
  terlepas dari pola pembelian aktual.

**Kapan pakai yang mana:**
- Gunakan **Association Rules** saat data transaksi historis melimpah dan
  ingin menangkap pola pembelian nyata (cocok untuk cross-selling, bundling promo).
- Gunakan **Content-Based** saat produk baru belum punya cukup data transaksi
  (cold-start problem), karena rekomendasi bisa langsung dibuat dari atribut produk.
- **Hybrid** (menggabungkan keduanya) ideal untuk sistem rekomendasi produksi
  nyata — Association Rules menangkap pola pembelian riil, sementara
  Content-Based menutupi kelemahan cold-start dan memperkaya variasi rekomendasi.